# OPT-HMM：OPT-1.3B + arXiv
Git 管理代码和配置，Colab 执行实验。按顺序运行单元格。GitHub token 每次通过隐藏输入框输入，不写入 Notebook、项目文件、远程 URL 或凭据缓存。

运行前，将本地最新代码推送到共享仓库。从第 4 步开始使用子进程管道和定期显示更新，处理换行及进度条的回车刷新；Colab 前端仍可能合并刷新，不承诺逐 token 显示。


## 第 1 步：选择 GPU
菜单“运行时 → 更改运行时类型”选择 GPU，再检查显卡。


In [ ]:
!nvidia-smi


## 第 2 步：克隆或更新私有仓库
在 [GitHub 创建只读令牌](https://github.com/settings/personal-access-tokens/new?name=Colab-HMM&target_name=kangqf4096&contents=read&expires_in=7)，选择 Resource owner `kangqf4096`、Only select repositories → HMM、Contents → Read-only。

首次运行下格会 clone；已有 checkout 时对当前分支执行 `pull --ff-only`。运行时粘贴 token 并回车。认证仅传入 Git 子进程，禁用凭据 helper，并在调用结束后清理临时变量。


In [ ]:
import base64
import os
import subprocess
from getpass import getpass
from pathlib import Path

def github_sync(root="/content/opt-hmm"):
    """Clone or fast-forward the current branch, prompting for a token each time."""
    project = Path(root).resolve()
    repo = "https://github.com/kangqf4096/HMM.git"
    if (project / ".git").is_dir():
        branch = subprocess.check_output(
            ["git", "-C", str(project), "branch", "--show-current"], text=True).strip()
        if not branch:
            raise RuntimeError("Checkout is detached. Switch to the intended branch before pulling.")
        command = ["git", "-C", str(project), "pull", "--ff-only", repo, branch]
    else:
        command = ["git", "clone", repo, str(project)]
    token = getpass("GitHub token（每次输入，不保存）: ").strip()
    if not token:
        raise ValueError("Token is empty.")
    auth = base64.b64encode(f"kangqf4096:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "2",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        "GIT_CONFIG_KEY_1": "credential.helper",
        "GIT_CONFIG_VALUE_1": "",
    })
    try:
        subprocess.run(command, env=git_env, check=True)
    finally:
        git_env.pop("GIT_CONFIG_VALUE_0", None)
        token = auth = None
    print("代码已同步：", project)
    return project

PROJECT = github_sync()
os.chdir(PROJECT)


## 第 3 步：安装并检查环境
安装到项目 `.venv`，让 vLLM 选择兼容依赖。首次安装可能较慢，检查成功后继续。


In [ ]:
%%bash
set -euo pipefail
cd /content/opt-hmm
source scripts/env.sh
if ! python3 -m venv .venv; then
    python3 -m pip install virtualenv
    python3 -m virtualenv .venv
fi
.venv/bin/python -m pip install -e ".[inference]"
.venv/bin/python scripts/test.py
.venv/bin/python scripts/run.py check-runtime


## 加载进度显示工具
第 4 步之前运行一次；pull 更新辅助代码后重新运行这一格。


In [ ]:
import runpy

tools = runpy.run_path("/content/opt-hmm/scripts/colab_tools.py")
run_step = tools["run_step"]
github_sync = tools["github_sync"]


## 第 4 步：下载 OPT-1.3B
固定版本的权重、配置和 tokenizer 下载到 `models/opt-1.3b`。权重约 2.63 GB。显示库报告的下载进度；重跑复用下载状态。


In [ ]:
model_log = run_step(
    "scripts/download_model.py",
    title="第 4 步：下载 OPT-1.3B", log_name="model",
)


## 第 5 步：准备 arXiv
缓存完整 `document` 配置（约 3.61 GB 下载、约 7.55 GB Arrow 数据），再选取 92 篇独立论文；输出 `data/ax0`。下载阶段显示库报告的进度，抽样阶段显示已选论文数。模型、依赖、缓存和结果需要额外磁盘空间。

只准备资源时，到此即可。输出目录必须为空；已成功准备后无需重跑。


In [ ]:
prepare_log = run_step(
    "scripts/run.py", "prepare",
    "--config", "configs/colab_arxiv.json",
    "--output", "data/ax0",
    title="第 5 步：准备 arXiv", log_name="prepare",
)


## 第 6 步：生成并评价
生成阶段显示 vLLM 的请求进度，随后核验结果；评价显示校准/推断任务数和 bootstrap 方法数。


In [ ]:
generation_log = run_step(
    "scripts/run.py", "generate",
    "--config", "configs/colab_arxiv.json",
    "--prompts", "data/ax0",
    "--output", "runs/ax0-gen",
    title="第 6 步：生成实验文本", log_name="generate",
)


In [ ]:
evaluation_log = run_step(
    "scripts/run.py", "evaluate",
    "--input", "runs/ax0-gen",
    "--output", "runs/ax0-eval",
    title="第 6 步：校准与评价", log_name="evaluate",
)


## 第 7 步：查看报告


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

display(Markdown(Path("/content/opt-hmm/runs/ax0-eval/report.md").read_text(encoding="utf-8")))


## 第 8 步：保存实验
`/content` 会随运行时回收而消失。下格打包实验输入和结果并下载；模型和完整语料缓存可按固定版本重新下载。


In [ ]:
import subprocess
from google.colab import files

subprocess.run([
    "tar", "-czf", "/content/ax0.tar.gz", "-C", "/content/opt-hmm",
    "data/ax0", "runs/ax0-gen", "runs/ax0-eval",
], check=True)
files.download("/content/ax0.tar.gz")


## 后续拉取更新（需要时单独运行）
每次运行下格都重新输入 token；在生成/评价结束后更新代码。它使用当前分支的 `--ff-only` 更新，分支分叉或更新会覆盖本地修改时停止，并显示 Git 的原因。模型、数据、缓存和结果被 Git 忽略，正常 pull 不会重新下载它们。

更新后重跑“加载进度显示工具”；依赖声明有变化时重跑第 3 步。输出已存在时，新实验使用 `ax1` 等短名称。此操作不会修改本地 Windows 仓库的 remote。


In [ ]:
PROJECT = github_sync()


## 查看完整日志（需要时）
日志分别在 `tmp/model.log`、`tmp/prepare.log`、`tmp/generate.log`、`tmp/evaluate.log`；同一步重跑时追加。没有可靠总量的阶段显示活动条及耗时，阶段百分比不代表整个实验的总进度。子进程失败会抛出异常，保留日志；点停止时尝试终止该步骤的进程组。


In [ ]:
print(Path("/content/opt-hmm/tmp/prepare.log").read_text(encoding="utf-8")[-12000:])
